# 74장 · EC2 배포 구성과 프록시 경로

외부에서 보낸 요청은 공개 포트와 프록시를 거쳐 내부 API에 도착합니다. 배포 선언에서 예상한 경로와 실제 응답을 구분해 확인합니다.

과정 안내 API의 EC2 배포 설정에서 외부 HTTPS 요청과 내부 API 요청의 경로를 확인합니다. 설정으로 확인한 항목과 실제 서버에서 접속해 검증할 항목을 구분합니다.

실행 환경은 Python 3.12.15입니다. 로컬 API 과제에는 FastAPI 0.135.0와 httpx 0.28.1를 사용합니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 공개 포트와 API의 내부 주소

배포 예제에서는 HTTPS를 받는 프록시만 호스트의 443번에 연결합니다. 프록시는 Compose 네트워크 안의 api:8000으로 전달하고, API는 내부 db:5432를 사용합니다. DB 접속 정보를 브라우저에 건네지 않습니다.


### 브라우저 주소와 프록시 내부 주소

<p>서비스 선언에서 ports를 가진 항목을 고르면 proxy만 남습니다. 브라우저는 공개 프록시로 접속하고, 프록시는 내부 네트워크에서 api:8000을 찾습니다. 브라우저가 사용하는 주소에 내부 서비스 이름을 넣으면 이 경계를 건너뛸 수 없습니다.</p>


### 443:443에서 양쪽 포트 읽기

<p>콜론 앞은 호스트 포트, 뒤는 프록시 컨테이너 포트입니다. 두 숫자가 우연히 같아도 두 위치는 서로 다릅니다. 예제에서 api와 db에는 ports가 없으므로 직접 공개 포트 목록에 나타나지 않습니다.</p>


### 셀 01 · 공개 포트 매핑 해석

services의 세 항목에서 ports가 있는 서비스를 골라냅니다. 이 설정에서 외부 요청을 가장 먼저 받도록 선언한 서비스의 이름이 출력됩니다.

EC2는 클라우드에서 빌려 쓰는 가상 컴퓨터입니다. 학생 브라우저에서 접속하는 주소와 그 컴퓨터 안에서 프록시가 API로 연결하는 주소는 서로 다른 접속점입니다. 연결표에서 브라우저가 출발점인 줄을 먼저 보세요.


In [1]:
services = {"proxy": {"ports": ["443:443"]}, "api": {}, "db": {}}
published = {name: spec["ports"] for name, spec in services.items() if "ports" in spec}
print(published)

{'proxy': ['443:443']}


{'proxy': ['443:443']}에서 확인되는 것은 포트 공개 선언입니다. EC2 보안 그룹, 프록시 프로세스, 도메인과 인증서의 상태는 이 딕셔너리에 들어 있지 않습니다.


### 선언을 읽는 셀과 접속을 시도하는 셀

<p>published의 출력은 Python 딕셔너리에서 뽑은 설정입니다. 선택 실습에서 HTTPS 주소를 직접 넣어 /health를 요청한 뒤에야 응답 상태와 application 값을 관찰할 수 있습니다.</p>


### 과제 02 · 프록시의 내부 API 주소

프록시 지시문에서 서비스 이름과 포트를 분리하세요. 프록시가 이 주소로 요청을 보낼 때 전송 구간에 쓰는 프로토콜도 출력하세요.


In [2]:
from urllib.parse import urlsplit
directive = "proxy_pass http://api:8000;"
# 주소 끝의 세미콜론을 제거한 뒤 scheme, hostname, port를 읽으세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from urllib.parse import urlsplit
directive = &quot;proxy_pass http://api:8000;&quot;
upstream = urlsplit(directive.split()[1].rstrip(&quot;;&quot;))
print(upstream.scheme, upstream.hostname, upstream.port)</code></pre>
<p>실행 결과</p><pre><code>http api 8000</code></pre>
<p>브라우저는 공개 도메인으로 HTTPS 요청을 보내고 프록시는 같은 Compose 네트워크의 api에 HTTP 요청을 보냅니다. 이 설정에서 TLS가 끝나는 곳은 프록시입니다.</p>
</details>


## 배포 주소와 내부 API 경로 검증

공개 주소가 있으면 외부에서 /health를 요청할 수 있습니다. 주소가 없는 경우에도 마지막 과제에서 로컬 API에 두 경로를 보내 어느 경로가 서비스 이름을 돌려주는지 확인할 수 있습니다.


### 셀 03 · 선택 실습 · 배포된 health 조회

수업에 배정된 HTTPS 주소가 있을 때만 base_url에 입력하세요. 로컬 실습 기본값으로는 요청을 보내지 않습니다. 서버 본문 전체 대신 HTTP 코드와 health의 application만 확인합니다.


In [3]:
import json
from urllib.parse import urlsplit
from urllib.request import urlopen
base_url = ""
if base_url:
    if urlsplit(base_url).scheme != "https":
        raise ValueError("수업 서비스의 HTTPS 주소를 넣으세요.")
    with urlopen(base_url.rstrip("/") + "/health", timeout=5) as reply:
        body = json.load(reply)
        print(reply.status, body.get("application"))
else:
    print("배정된 배포 주소가 없어 HTTPS 확인을 건너뜁니다.")

배정된 배포 주소가 없어 HTTPS 확인을 건너뜁니다.


기본값의 안내는 미실행을 뜻합니다. 실제 요청에서 200이 와도 앞서 저장한 문서가 옮겨졌는지, 답변이 같은 근거를 인용하는지는 해당 요청으로 별도 확인해야 합니다. 수업에서 배정한 HTTPS 주소를 base_url에 넣고 /health 응답의 상태 코드와 application 값을 함께 적으세요. 주소가 없는 상태에서 설정을 읽은 경우 확인한 값은 포트와 프록시 대상까지입니다.


### HTTPS 검사와 /health 요청

<p>빈 주소를 넣으면 요청을 건너뜁니다. 주소를 넣었지만 scheme이 https가 아니면 ValueError가 먼저 발생합니다. HTTPS 주소인 경우에만 /health로 연결하므로, 주소 형식 오류와 네트워크 오류를 구분할 수 있습니다.</p>


### 과제 04 · 프록시가 넘긴 경로와 API 응답

프록시에서 내부 API로 넘길 경로에 /api 접두어가 그대로 남으면 어떻게 될까요? 두 경로를 같은 로컬 API에 보내 응답을 비교하세요. 노트북 안의 TestClient가 호출을 전달하므로 실제 프록시·인증서·EC2는 필요하지 않습니다.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

internal_api = FastAPI()
@internal_api.get("/health")
def health():
    return {"application": "course-api"}

upstream_client = TestClient(internal_api)
# /api/health와 /health에 GET을 보내 두 상태 코드와 정상 응답의 서비스 이름을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>from fastapi import FastAPI
from fastapi.testclient import TestClient

internal_api = FastAPI()
@internal_api.get(&quot;/health&quot;)
def health():
    return {&quot;application&quot;: &quot;course-api&quot;}

upstream_client = TestClient(internal_api)

with upstream_client:
    wrong = upstream_client.get(&quot;/api/health&quot;)
    forwarded = upstream_client.get(&quot;/health&quot;)
print(wrong.status_code, forwarded.status_code, forwarded.json()[&quot;application&quot;])</code></pre>
<p>실행 결과</p><pre><code>404 200 course-api</code></pre>
<p>같은 앱이라도 전달된 경로가 다르면 첫 요청은 등록된 라우트를 찾지 못합니다. 두 번째 요청에서는 응답 본문까지 읽어 어느 앱이 답했는지 확인했습니다. 실제 프록시가 접두어를 어떻게 처리하는지는 프록시 설정과 요청 기록으로 확인합니다.</p>
</details>


## 돌아보기

외부 주소가 열리지 않을 때 포트 선언만 보고 원인을 확정할 수 없습니다. 반대로 /health가 200이어도 DB를 쓰는 기능은 실패할 수 있습니다. 두 경우에는 각각 실제 HTTPS 응답과 기존 문서 조회 결과가 필요합니다.
